***PART 1 Two-Layer Neural Network (Manual Implementation)***

In [ ]:
import torch

In [ ]:
def f(x):
  return x[:,3:4]**4 + x[:,0:1]**3 + 3*x[:,1:2]**2 + 3*x[:,4:5] + torch.rand(x.size(0),1)

In [ ]:
x = torch.tensor([
 [1.0, 2.1, 3.2, 4.3, 5.4],
 [2.2, 1.5, 0.9, 3.3, 4.1],
 [3.1, 2.2, 1.8, 0.5, 2.6],
 [4.0, 3.3, 2.7, 1.1, 0.4],
 [5.5, 4.4, 3.3, 2.2, 1.1],
 [6.1, 5.0, 4.2, 3.1, 2.0],
 [7.3, 6.2, 5.1, 4.0, 3.3],
 [8.0, 7.1, 6.0, 5.2, 4.1],
 [9.4, 8.3, 7.2, 6.1, 5.0],
 [10.2, 9.1, 8.0, 7.3, 6.4],
 [11.0, 10.5, 9.2, 8.1, 7.0],
 [12.3, 11.2, 10.1, 9.0, 8.5],
 [13.1, 12.0, 11.3, 10.2, 9.4],
 [14.5, 13.4, 12.2, 11.1, 10.0],
 [15.0, 14.2, 13.1, 12.3, 11.2]
])


Lets take first row as an example.The function will take x[3] and do
x[3]^4 = (4.3)^4.Other values will calculated by doing some opreration defined in f(x).Finally the addition of these numbers along with some random number will be stored as y_true.

In [ ]:
y_true_small=f(x)

In [ ]:
""" we have 5 input features and we want to create hidden
 layer of 10 neurons
 Also we want 1 output for now..We may require more than i output some times
 """

W1_manual=torch.rand(5,10)
b1_manual=torch.rand(1,10)

W2_manual=torch.rand(10,1)
b2_manual=torch.rand(1)



In [ ]:
for k in range(100):
  # forward pass to calculate Z1
  # first layer of N.This layer is hidden layer
  z1= x @ W1_manual + b1_manual
  h=torch.relu(z1)
  # second layer ,the output layer
  y_pred= h @ W2_manual + b2_manual

  error=y_pred-y_true_small
  loss=((error)**2).mean()
  n=x.size(0)
  dW2=(2/n)*(h.T @ error )
  db2=(2/n)*error.sum(dim=0)

  """now we want to change the weights of the hidden layer but will need the
    gradient of last layer which is known as backward pass..The gradient is '1' for
  relu if the input is > 0 else 0,"""
  relu_grad = (z1 > 0).float()
  dW1 = (2/n) * (x.T @ (error @ W2_manual.T * relu_grad))
  db1 = (2/n) * (error @ W2_manual.T * relu_grad).sum(dim=0)

  lr=1
  W2_manual = W2_manual - lr*dW2
  b2_manual = b2_manual - lr*db2
  W1_manual = W1_manual - lr * dW1
  b1_manual = b1_manual - lr * db1

# **Part 2: Same Network Using Autograd**

In [ ]:
W1=torch.rand((5,10),requires_grad=True)
b1=torch.rand((1,10),requires_grad=True)

W2=torch.rand((10,1),requires_grad=True)
b2=torch.rand((1,),requires_grad=True)

In [ ]:
lr=0.1
for k in range(100):
  # forward pass to calculate Z1
  # first layer of N.This layer is hidden layer
  z1= x @ W1 + b1
  h=torch.relu(z1)
  # second layer ,the output layer
  y_pred= h @ W2 + b2

  error=y_pred-y_true_small
  loss=((error)**2).mean()

  loss.backward()


  with torch.no_grad():
      W1 -=  lr * W1.grad
      b1 -=  lr * b1.grad
      W2 -=  lr * W2.grad
      b2 -=  lr * b2.grad

  W2.grad=None
  b2.grad=None
  W1.grad=None
  b1.grad=None

# Part 3: Fully Connected Network for Image Data (MNIST-type)

In [ ]:
from torch.utils.data import Dataset
from torchvision import datasets,transforms
import torch.nn as nn

my_transforms=transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x : torch.flatten(x))
])

In [ ]:
train_dataset=datasets.MNIST(
    root="./data",
    train=True,
    transform=my_transforms,
    download=True
)

test_dataset=datasets.MNIST(
    root="./data",
    train=False,
    transform=my_transforms,
    download=True
)
# X_train, y_true = train_dataset[0] #single sample
# print(X_train.shape)

X_train=torch.stack([train_dataset[i][0] for i in range(len(train_dataset))])
y_true=train_dataset.targets

X_test=torch.stack([test_dataset[i][0] for i in range(len(test_dataset))])
y_true_test=test_dataset.targets

100%|██████████| 9.91M/9.91M [00:00<00:00, 14.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 342kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.22MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.72MB/s]


In [ ]:
import torch
import torch.nn as nn

W1=nn.Parameter(torch.randn((784,64))*0.01)
b1=nn.Parameter(torch.randn((64,))*0.01)
W2=nn.Parameter(torch.randn((64,32))*0.01)
b2=nn.Parameter(torch.randn((32,))*0.01)
W3=nn.Parameter(torch.randn((32,10))*0.01)
b3=nn.Parameter(torch.randn((10,))*0.01)

lr=0.01

In [ ]:
# timeToExecute
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

https://discuss.pytorch.org/t/how-to-measure-time-in-pytorch/26964

In [ ]:
loss_fn=nn.CrossEntropyLoss()
start.record()
batch_size=50

for i in range(10):
  for batch_start in range(0, len(X_train), batch_size):
      # Zero gradients at the beginning of each batch processing
      for param in [W1, b1, W2, b2, W3, b3]:
          if param.grad is not None:
              param.grad.zero_()

      X_batch = X_train[batch_start:batch_start+batch_size].clone().detach()
      y_batch = y_true[batch_start:batch_start+batch_size].clone().detach()

      z1=X_batch @ W1 + b1
      h1=torch.relu(z1)
      z2=h1 @ W2 + b2
      h2=torch.relu(z2)
      y_pred=h2 @ W3 + b3

      loss=loss_fn(y_pred,y_batch)
      loss.backward() # Removed retain_graph=True

      with torch.no_grad():
          W1.data.sub_(lr * W1.grad)
          b1.data.sub_(lr * b1.grad)
          W2.data.sub_(lr * W2.grad)
          b2.data.sub_(lr * b2.grad)
          W3.data.sub_(lr * W3.grad)
          b3.data.sub_(lr * b3.grad)

      if i%10 == 0 :
        print(f"loss : {loss}")

end.record()
# Waits for everything to finish running
torch.cuda.synchronize()
TimeToExecute = start.elapsed_time(end)

loss : 2.303598403930664
loss : 2.3016202449798584
loss : 2.3023953437805176
loss : 2.3059656620025635
loss : 2.3011531829833984
loss : 2.302456855773926
loss : 2.3044447898864746
loss : 2.3041627407073975
loss : 2.3031439781188965
loss : 2.3023056983947754
loss : 2.301114320755005
loss : 2.3049581050872803
loss : 2.303542375564575
loss : 2.3019776344299316
loss : 2.301220178604126
loss : 2.3020195960998535
loss : 2.3040289878845215
loss : 2.302647352218628
loss : 2.304321050643921
loss : 2.3025269508361816
loss : 2.3013603687286377
loss : 2.301189661026001
loss : 2.3045411109924316
loss : 2.303823947906494
loss : 2.301724672317505
loss : 2.302626609802246
loss : 2.305166721343994
loss : 2.3045194149017334
loss : 2.30336856842041
loss : 2.3006622791290283
loss : 2.301994800567627
loss : 2.3054583072662354
loss : 2.3034141063690186
loss : 2.3023672103881836
loss : 2.3042514324188232
loss : 2.302074909210205
loss : 2.3021743297576904
loss : 2.305746078491211
loss : 2.303196430206299
loss

In [ ]:
z1=X_test @ W1 + b1;
h1=torch.relu(z1)
z2=h1 @ W2 + b2
h2=torch.relu(z2)
y_pred_test=h2 @ W3 + b3

test_loss=loss_fn(y_pred_test,y_true_test)

correct=(y_pred_test.argmax(dim=1)==y_true_test).sum().item()
total=y_pred_test.size(0)

Accuracy=correct/total
print(Accuracy)
print(loss.item())

0.8736
0.28783681988716125


#***Part 4: Implementing commonly used architecture***

In [ ]:
import torch
import torch.nn as nn

W1_common=nn.Parameter(torch.randn((784,256))*0.01)
b1_common=nn.Parameter(torch.zeros((256,)))
W2_common=nn.Parameter(torch.randn((256,128))*0.01)
b2_common=nn.Parameter(torch.zeros((128,)))
W3_common=nn.Parameter(torch.randn((128,10))*0.01)
b3_common=nn.Parameter(torch.zeros((10,)))


start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

In [ ]:
lr=0.01
start.record()
batch_size=50

loss_fn=nn.CrossEntropyLoss() # Moved loss_fn definition outside the loop

for i in range(10):
  for batch_start in range(0, len(X_train), batch_size):

      # Zero gradients at the beginning of each batch processing
      for param in [W1_common, b1_common, W2_common, b2_common, W3_common, b3_common]:
          if param.grad is not None:
              param.grad.zero_()

      X_batch = X_train[batch_start:batch_start+batch_size]
      y_batch = y_true[batch_start:batch_start+batch_size]

      z1=X_batch @ W1_common + b1_common
      h1=torch.relu(z1)
      z2=h1 @ W2_common + b2_common
      h2=torch.relu(z2)
      y_pred_common=h2 @ W3_common + b3_common

      loss=loss_fn(y_pred_common,y_batch)
      loss.backward()

      with torch.no_grad(): # Ensure updates are not part of the computation graph
          W1_common.data.sub_(lr * W1_common.grad)
          b1_common.data.sub_(lr * b1_common.grad)
          W2_common.data.sub_(lr * W2_common.grad)
          b2_common.data.sub_(lr * b2_common.grad)
          W3_common.data.sub_(lr * W3_common.grad)
          b3_common.data.sub_(lr * b3_common.grad)


  if i % 1 == 0 : # Print loss after each epoch
      print(f"loss : {loss.item()}") # Use .item() for scalar loss


end.record()

torch.cuda.synchronize()
TimeToExecuteCommonlyUsed = start.elapsed_time(end)

loss : 2.297175645828247
loss : 2.0454201698303223
loss : 0.6235935091972351
loss : 0.42453476786613464
loss : 0.3140368163585663
loss : 0.23832014203071594
loss : 0.19125761091709137
loss : 0.16068784892559052
loss : 0.1381026953458786
loss : 0.11990735679864883


In [ ]:
z1=X_test @ W1_common + b1_common;
h1=torch.relu(z1)
z2=h1 @ W2_common + b2_common
h2=torch.relu(z2)
y_pred_test=h2 @ W3_common + b3_common

test_loss=loss_fn(y_pred_test,y_true_test)

correct=(y_pred_test.argmax(dim=1)==y_true_test).sum().item()
total=y_pred_test.size(0)

Accuracy=correct/total


In [ ]:
print(Accuracy)
print(loss.item())
print(TimeToExecuteCommonlyUsed)
print(TimeToExecute)

0.9171
0.11990735679864883
19245.7421875
10533.7578125


*commonly used artitecture takes about 70% more time to train which is quite logical
 as the neural network used has very high amount of neurons compared to our neural network*

### Part 5: Transfer Learning and Freezing Layers

In [ ]:
import torch.nn as nn
import torch.optim as optim

# Define the neural network model as a class
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, num_classes):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size2, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu1(out)
        out = self.fc2(out)
        out = self.relu2(out)
        out = self.fc3(out)
        return out

# Instantiate the model with the architecture from Part 4
input_size = 784
hidden_size1 = 256
hidden_size2 = 128
num_classes = 10 # MNIST has 10 classes

model = SimpleNN(input_size, hidden_size1, hidden_size2, num_classes)

print(model)

SimpleNN(
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=128, out_features=10, bias=True)
)


#### Freeze earlier layers and train only the final layer

In [ ]:
# Freeze all parameters initially
for param in model.parameters():
    param.requires_grad = False

# Unfreeze only the final layer's parameters (fc3)
model.fc3.weight.requires_grad = True
model.fc3.bias.requires_grad = True

# Verify which parameters are trainable
print("\nTrainable parameters after freezing:")
for name, param in model.named_parameters():
    print(f"{name}: {param.requires_grad}")

# Define loss function and optimizer for training the final layer
loss_fn = nn.CrossEntropyLoss()
lr_frozen = 0.01
optimizer_frozen = optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), lr=lr_frozen)

# Training loop for the frozen layers
start_frozen = torch.cuda.Event(enable_timing=True)
end_frozen = torch.cuda.Event(enable_timing=True)

epochs_frozen = 5
batch_size = 50 # Using the same batch size as before

start_frozen.record()

print(f"\nTraining with frozen layers for {epochs_frozen} epochs...")
for epoch in range(epochs_frozen):
    for batch_start in range(0, len(X_train), batch_size):
        X_batch = X_train[batch_start:batch_start+batch_size]
        y_batch = y_true[batch_start:batch_start+batch_size]

        optimizer_frozen.zero_grad()

        outputs = model(X_batch)
        loss = loss_fn(outputs, y_batch)
        loss.backward()
        optimizer_frozen.step()

    print(f'Epoch [{epoch+1}/{epochs_frozen}], Loss: {loss.item():.4f}')

end_frozen.record()
torch.cuda.synchronize()
TimeToExecuteFrozen = start_frozen.elapsed_time(end_frozen)
print(f"Training time with frozen layers: {TimeToExecuteFrozen:.2f} ms")

# Evaluate performance after training final layer
with torch.no_grad():
    model.eval()
    outputs_test_frozen = model(X_test)
    loss_test_frozen = loss_fn(outputs_test_frozen, y_true_test)
    correct_frozen = (outputs_test_frozen.argmax(dim=1) == y_true_test).sum().item()
    total_test = y_true_test.size(0)
    accuracy_frozen = correct_frozen / total_test
    print(f'Accuracy after training frozen layers: {accuracy_frozen:.4f}')
    model.train() # Set model back to training mode


Trainable parameters after freezing:
fc1.weight: False
fc1.bias: False
fc2.weight: False
fc2.bias: False
fc3.weight: True
fc3.bias: True

Training with frozen layers for 5 epochs...
Epoch [1/5], Loss: 2.2788
Epoch [2/5], Loss: 2.2580
Epoch [3/5], Loss: 2.2382
Epoch [4/5], Loss: 2.2188
Epoch [5/5], Loss: 2.1999
Training time with frozen layers: 5220.70 ms
Accuracy after training frozen layers: 0.5163


#### Unfreeze all layers and retrain the entire network

In [ ]:
# Unfreeze all parameters
for param in model.parameters():
    param.requires_grad = True

# Verify all parameters are now trainable
print("\nTrainable parameters after unfreezing:")
for name, param in model.named_parameters():
    print(f"{name}: {param.requires_grad}")

# Define a new optimizer for the entire network
lr_unfrozen = 0.001 # Often a smaller learning rate is used for fine-tuning
optimizer_unfrozen = optim.SGD(model.parameters(), lr=lr_unfrozen)

# Training loop for the unfrozen network
start_unfrozen = torch.cuda.Event(enable_timing=True)
end_unfrozen = torch.cuda.Event(enable_timing=True)

epochs_unfrozen = 5

start_unfrozen.record()

print(f"\nTraining with unfrozen layers for {epochs_unfrozen} epochs...")
for epoch in range(epochs_unfrozen):
    for batch_start in range(0, len(X_train), batch_size):
        X_batch = X_train[batch_start:batch_start+batch_size]
        y_batch = y_true[batch_start:batch_start+batch_size]

        optimizer_unfrozen.zero_grad()

        outputs = model(X_batch)
        loss = loss_fn(outputs, y_batch)
        loss.backward()
        optimizer_unfrozen.step()

    print(f'Epoch [{epoch+1}/{epochs_unfrozen}], Loss: {loss.item():.4f}')

end_unfrozen.record()
torch.cuda.synchronize()
TimeToExecuteUnfrozen = start_unfrozen.elapsed_time(end_unfrozen)
print(f"Training time with unfrozen layers: {TimeToExecuteUnfrozen:.2f} ms")

# Evaluate final performance
with torch.no_grad():
    model.eval()
    outputs_test_unfrozen = model(X_test)
    loss_test_unfrozen = loss_fn(outputs_test_unfrozen, y_true_test)
    correct_unfrozen = (outputs_test_unfrozen.argmax(dim=1) == y_true_test).sum().item()
    accuracy_unfrozen = correct_unfrozen / total_test
    print(f'Accuracy after training unfrozen layers: {accuracy_unfrozen:.4f}')
    model.train()


Trainable parameters after unfreezing:
fc1.weight: True
fc1.bias: True
fc2.weight: True
fc2.bias: True
fc3.weight: True
fc3.bias: True

Training with unfrozen layers for 5 epochs...
Epoch [1/5], Loss: 1.7605
Epoch [2/5], Loss: 1.2830
Epoch [3/5], Loss: 0.9532
Epoch [4/5], Loss: 0.7493
Epoch [5/5], Loss: 0.6155
Training time with unfrozen layers: 9330.43 ms
Accuracy after training unfrozen layers: 0.8341


#### Comparison

In [ ]:
print("\n--- Performance and Speed Comparison ---")
print(f"Accuracy after training only final layer: {accuracy_frozen:.4f}")
print(f"Training time (final layer only): {TimeToExecuteFrozen:.2f} ms")
print(f"Accuracy after unfreezing and retraining: {accuracy_unfrozen:.4f}")
print(f"Training time (unfrozen network): {TimeToExecuteUnfrozen:.2f} ms")

# Original model's accuracy and time for comparison from Part 4
print(f"Original Part 4 Model Accuracy: {Accuracy:.4f}")
print(f"Original Part 4 Model Training Time: {TimeToExecuteCommonlyUsed:.2f} ms")


--- Performance and Speed Comparison ---
Accuracy after training only final layer: 0.5163
Training time (final layer only): 5220.70 ms
Accuracy after unfreezing and retraining: 0.8341
Training time (unfrozen network): 9330.43 ms
Original Part 4 Model Accuracy: 0.9171
Original Part 4 Model Training Time: 19245.74 ms
